In [1]:
# 필요 라이브러리 임포트
import pandas as pd
import mlflow
import mlflow.xgboost
import mlflow.pyfunc
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

# 1. 데이터 로딩 및 전처리
data = pd.read_csv('../data/churn.csv')  # 고객 이탈 데이터 로딩


In [2]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [3]:

# 사용하지 않는 열 제거 및 feature/target 분리
X = data.drop(['Exited', 'RowNumber', 'CustomerId', 'Surname'], axis=1)
y = data['Exited']

In [4]:
# 범주형 및 수치형 feature 정의
categorical = ['Geography', 'Gender']
numeric = ['CreditScore', 'Age', 'Tenure', 'Balance',
           'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']


In [5]:
# 전처리 파이프라인: 수치형 표준화 + 범주형 원-핫 인코딩
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric),
    ('cat', OneHotEncoder(), categorical)
])

In [6]:
# 전처리 적용
X_processed = preprocessor.fit_transform(X)



In [7]:
# 학습/테스트 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42
)

In [8]:

# 2. 하이퍼파라미터 탐색 공간 정의 (Hyperopt 사용)
space = {
    'max_depth': hp.choice('max_depth', range(3, 10)),               # 트리 깊이
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),         # 학습률
    'n_estimators': hp.choice('n_estimators', range(50, 200)),       # 트리 개수
    'gamma': hp.uniform('gamma', 0, 5)                                # 가지치기 규제 파라미터
}

In [9]:
# 3. objective 함수 정의: 실험 수행 + 성능 평가 + 조건 충족 시 모델 자동 등록
def objective(params):
    mlflow.set_experiment("practice4")  # 실험 이름 설정
    with mlflow.start_run(nested=True) as run:  # 각 하이퍼파라미터 조합마다 별도 Run 시작
        # 모델 학습
        model = xgb.XGBClassifier(eval_metric='logloss', **params)
        model.fit(X_train, y_train)

        # ROC-AUC 계산을 위한 확률 예측
        probs = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, probs)

        # 실험 결과 로깅 (하이퍼파라미터 및 성능 지표)
        mlflow.log_params(params)
        mlflow.log_metric("roc_auc", roc_auc)

        # 조건 만족 시 (성능 기준 통과), 모델 Registry에 자동 등록 및 Staging 전환
        if roc_auc > 0.86:
            # 모델 시그니처 정의
            signature = infer_signature(X_train, probs)

            # 모델 저장 및 등록
            result = mlflow.xgboost.log_model(
                model,
                artifact_path="model",
                signature=signature,
                registered_model_name="CICDModel"
            )

            # Registry에 등록된 모델 버전 확인 후, Staging 단계로 전환
            client = MlflowClient()
            latest_version = client.get_latest_versions("CICDModel", stages=["None"])[0].version
            client.transition_model_version_stage(
                name="CICDModel",
                version=latest_version,
                stage="Staging",
                archive_existing_versions=True
            )

            print(f"등록 완료: version {latest_version}, AUC={roc_auc:.4f}")

        # Hyperopt가 사용하는 반환 형식 (최소화 대상: -roc_auc)
        return {'loss': -roc_auc, 'status': STATUS_OK}


In [10]:
# 4. 하이퍼파라미터 최적화 수행 (총 30회 시도)
trials = Trials()
best = fmin(
    fn=objective,        # 목적 함수
    space=space,         # 탐색 공간
    algo=tpe.suggest,    # 탐색 알고리즘: TPE
    max_evals=30,        # 총 30회 시도
    trials=trials        # 실험 결과 저장 객체
)

  0%|          | 0/30 [00:00<?, ?trial/s, best loss=?]

2026/04/22 11:51:32 INFO mlflow.tracking.fluent: Experiment with name 'practice4' does not exist. Creating a new experiment.

c:\Users\SSAFY\Desktop\TIL\09_data_analysis\g_mlflow\mlflow_env\Lib\site-packages\xgboost\sklearn.py:1028: UserWarning: [11:51:33] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)

Successfully registered model 'CICDModel'.
2026/04/22 11:51:41 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: CICDModel, version 1



등록 완료: version 1, AUC=0.8748                          
🏃 View run thoughtful-panda-853 at: http://127.0.0.1:5000/#/experiments/10/runs/05cd85fb88f944c2aa289639dad9b06e

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/10

  3%|▎         | 1/30 [00:09<04:28,  9.26s/trial, best loss: -0.8747844592123201]

Created version '1' of model 'CICDModel'.
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_32020\360130756.py:32: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_version = client.get_latest_versions("CICDModel", stages=["None"])[0].version

C:\Users\SSAFY\AppData\Local\Temp\ipykernel_32020\360130756.py:33: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(

c:\Users\SSAFY\Desk

등록 완료: version 2, AUC=0.8669                                                     
🏃 View run dazzling-tern-294 at: http://127.0.0.1:5000/#/experiments/10/runs/4839d2afeccb455fb84bad1bea9e16d5

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/10                    

  7%|▋         | 2/30 [00:13<02:48,  6.02s/trial, best loss: -0.8747844592123201]

...

Created version '25' of model 'CICDModel'.
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_32020\360130756.py:32: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_version = client.get_latest_versions("CICDModel", stages=["None"])[0].version

C:\Users\SSAFY\AppData\Local\Temp\ipykernel_32020\360130756.py:33: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(

c:\Users\SSAFY\Des

등록 완료: version 26, AUC=0.8744                                                     
🏃 View run clumsy-shrimp-688 at: http://127.0.0.1:5000/#/experiments/10/runs/e8610e84dd534ae1aa9ed517bc69107e

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/10                     

 97%|█████████▋| 29/30 [01:42<00:03,  3.51s/trial, best loss: -0.8758176299301244]

Created version '26' of model 'CICDModel'.
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_32020\360130756.py:32: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_version = client.get_latest_versions("CICDModel", stages=["None"])[0].version

C:\Users\SSAFY\AppData\Local\Temp\ipykernel_32020\360130756.py:33: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(

c:\Users\SSAFY\Des

등록 완료: version 27, AUC=0.8716                                                     
🏃 View run aged-stag-590 at: http://127.0.0.1:5000/#/experiments/10/runs/365a6b7dcd3d49778f847e5ca5bceafb

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/10                     

100%|██████████| 30/30 [01:46<00:00,  3.55s/trial, best loss: -0.8758176299301244]


Created version '27' of model 'CICDModel'.
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_32020\360130756.py:32: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_version = client.get_latest_versions("CICDModel", stages=["None"])[0].version

C:\Users\SSAFY\AppData\Local\Temp\ipykernel_32020\360130756.py:33: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(

